# 01 数据爬取 - 中国主要城市公共预算收支数据

从国家统计局 API 爬取主要城市年度数据，包括：
- 地方一般公共预算收入
- 地方一般公共预算支出
- 住户存款余额
- 地区生产总值（GDP）
- 房地产相关数据

In [1]:
import requests
import json
import pandas as pd
import time
import os
import urllib3
urllib3.disable_warnings()

DATA_RAW = 'data_raw'
DATA_CLEAN = 'data_clean'
OUTPUT = 'output'

for d in [DATA_RAW, DATA_CLEAN, OUTPUT]:
    os.makedirs(d, exist_ok=True)

print('Folders ready.')

Folders ready.


In [2]:
session = requests.Session()
session.verify = False
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/json, text/javascript, */*; q=0.01',
    'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8',
    'Referer': 'https://data.stats.gov.cn/easyquery.htm?cn=E0105',
    'X-Requested-With': 'XMLHttpRequest',
})

BASE_URL = 'https://data.stats.gov.cn/easyquery.htm'

# 先访问页面获取 cookies
session.get('https://data.stats.gov.cn/easyquery.htm?cn=E0105', timeout=15)
print('Session ready.')

Session ready.


In [3]:
def fetch_data(zb_code, time_range='LAST20'):
    """从国家统计局 API 获取指定指标的城市年度数据"""
    params = {
        'm': 'QueryData',
        'dbcode': 'csnd',
        'rowcode': 'reg',
        'colcode': 'sj',
        'wds': json.dumps([{'wdcode': 'zb', 'valuecode': zb_code}]),
        'dfwds': json.dumps([{'wdcode': 'sj', 'valuecode': time_range}]),
    }
    resp = session.get(BASE_URL, params=params, timeout=30)
    data = resp.json()
    
    if data.get('returncode') != 200:
        print(f'Error fetching {zb_code}: {data}')
        return pd.DataFrame()
    
    rd = data['returndata']
    
    # 提取城市代码和名称映射
    city_map = {}
    for wn in rd.get('wdnodes', []):
        if wn['wdcode'] == 'reg':
            for node in wn['nodes']:
                if node:
                    city_map[node['code']] = node['cname']
    
    # 提取数据
    rows = []
    for dn in rd.get('datanodes', []):
        if not dn['data']['hasdata']:
            continue
        wds = {w['wdcode']: w['valuecode'] for w in dn['wds']}
        rows.append({
            'city_code': wds.get('reg', ''),
            'city': city_map.get(wds.get('reg', ''), ''),
            'year': int(wds.get('sj', 0)),
            'value': dn['data']['data']
        })
    
    df = pd.DataFrame(rows)
    print(f'  {zb_code}: {len(df)} records, cities: {df["city"].nunique()}, years: {sorted(df["year"].unique())[:3]}...{sorted(df["year"].unique())[-3:]}')
    return df

In [4]:
# 指标定义
indicators = {
    'A0101': ('gdp', '地区生产总值(亿元)'),
    'A0401': ('income', '地方一般公共预算收入(亿元)'),
    'A0402': ('expend', '地方一般公共预算支出(亿元)'),
    'A0403': ('deposit', '住户存款余额(亿元)'),
}

# 房地产指标
re_indicators = {
    'A0302': ('re_invest', '房地产开发投资额(亿元)'),
    'A0309': ('re_sales_area', '商品房销售面积(万平方米)'),
    'A030B': ('re_avg_price', '商品房平均销售价格(元/平方米)'),
}

print('开始爬取主要财政指标...')
dfs = {}
for code, (name, desc) in indicators.items():
    print(f'正在获取: {desc}')
    df = fetch_data(code)
    df = df.rename(columns={'value': name})
    dfs[name] = df
    time.sleep(1)

print('\n开始爬取房地产指标...')
for code, (name, desc) in re_indicators.items():
    print(f'正在获取: {desc}')
    df = fetch_data(code)
    df = df.rename(columns={'value': name})
    dfs[name] = df
    time.sleep(1)

print('\n数据爬取完成！')

开始爬取主要财政指标...
正在获取: 地区生产总值(亿元)
  A0101: 682 records, cities: 36, years: [np.int64(2006), np.int64(2007), np.int64(2008)]...[np.int64(2022), np.int64(2023), np.int64(2024)]
正在获取: 地方一般公共预算收入(亿元)
  A0401: 682 records, cities: 36, years: [np.int64(2006), np.int64(2007), np.int64(2008)]...[np.int64(2022), np.int64(2023), np.int64(2024)]
正在获取: 地方一般公共预算支出(亿元)
  A0402: 682 records, cities: 36, years: [np.int64(2006), np.int64(2007), np.int64(2008)]...[np.int64(2022), np.int64(2023), np.int64(2024)]
正在获取: 住户存款余额(亿元)
  A0403: 679 records, cities: 36, years: [np.int64(2006), np.int64(2007), np.int64(2008)]...[np.int64(2022), np.int64(2023), np.int64(2024)]

开始爬取房地产指标...
正在获取: 房地产开发投资额(亿元)
  A0302: 666 records, cities: 36, years: [np.int64(2006), np.int64(2007), np.int64(2008)]...[np.int64(2022), np.int64(2023), np.int64(2024)]
正在获取: 商品房销售面积(万平方米)
  A0309: 665 records, cities: 35, years: [np.int64(2006), np.int64(2007), np.int64(2008)]...[np.int64(2022), np.int64(2023), np.int64(2024)]
正在获取: 商品房平均

In [5]:
# 保存原始数据到 xlsx
file_mapping = {
    'income': 'city_income.xlsx',
    'expend': 'city_expenditure.xlsx',
    'deposit': 'individual_deposit.xlsx',
    'gdp': 'gdp.xlsx',
    're_invest': 're_invest.xlsx',
    're_sales_area': 're_sales_area.xlsx',
    're_avg_price': 're_avg_price.xlsx',
}

for name, filename in file_mapping.items():
    if name in dfs and len(dfs[name]) > 0:
        filepath = os.path.join(DATA_RAW, filename)
        dfs[name].to_excel(filepath, index=False)
        print(f'已保存: {filepath} ({len(dfs[name])} rows)')

print('\n原始数据保存完毕！')

已保存: data_raw\city_income.xlsx (682 rows)
已保存: data_raw\city_expenditure.xlsx (682 rows)
已保存: data_raw\individual_deposit.xlsx (679 rows)
已保存: data_raw\gdp.xlsx (682 rows)
已保存: data_raw\re_invest.xlsx (666 rows)
已保存: data_raw\re_sales_area.xlsx (665 rows)
已保存: data_raw\re_avg_price.xlsx (665 rows)

原始数据保存完毕！


In [6]:
# 合并数据
merge_keys = ['city_code', 'city', 'year']

# 从 income 开始合并
merged = dfs['income'][merge_keys + ['income']].copy()

for name in ['expend', 'gdp', 'deposit']:
    if name in dfs and len(dfs[name]) > 0:
        merged = merged.merge(
            dfs[name][merge_keys + [name]],
            on=merge_keys,
            how='outer'
        )

# 合并房地产数据
for name in ['re_invest', 're_sales_area', 're_avg_price']:
    if name in dfs and len(dfs[name]) > 0:
        merged = merged.merge(
            dfs[name][merge_keys + [name]],
            on=merge_keys,
            how='outer'
        )

merged = merged.sort_values(['city_code', 'year']).reset_index(drop=True)

print(f'合并后数据: {merged.shape}')
print(f'城市数: {merged["city"].nunique()}')
print(f'年份范围: {merged["year"].min()} - {merged["year"].max()}')
merged.head(10)

合并后数据: (684, 10)
城市数: 36
年份范围: 2006 - 2024


,city_code,city,year,income,expend,gdp,deposit,re_invest,re_sales_area,re_avg_price
0,110000,北京,2006,1117.1514,1296.8389,8618.9,8705.600000,1719.8650,2607.6178,8279.510134
1,110000,北京,2007,1492.6380,1649.5023,10730.4,9155.340000,1995.8206,2176.5727,11553.258478
2,110000,北京,2008,1837.3238,1959.2857,12167.6,11952.840000,1908.7387,1335.3700,12418.000000
3,110000,北京,2009,2026.8089,2319.3658,13335.7,14672.100000,2337.7124,2362.2500,13799.000000
4,110000,北京,2010,2353.9301,2717.3174,15420.2,17003.112186,2901.0700,1639.5300,17782.000000
5,110000,北京,2011,3006.2800,3245.2300,17653.3,19126.136991,3036.3118,1439.2018,16851.952937
6,110000,北京,2012,3314.9340,3685.3076,19568.3,21644.900000,3153.4419,1943.7367,17021.632611
7,110000,北京,2013,3661.1097,4173.6563,21746.0,23086.414992,3483.4045,1903.1124,18553.000000
8,110000,北京,2014,4027.1609,4524.6690,23577.5,24158.400000,3715.3341,1454.1935,18833.000000
9,110000,北京,2015,4723.8600,5737.7011,26034.1,23913.967000,4177.0481,1554.2463,22633.000000


In [7]:
# 保存合并后的清洗数据
clean_path = os.path.join(DATA_CLEAN, 'merged_data.xlsx')
merged.to_excel(clean_path, index=False)
print(f'已保存清洗数据: {clean_path}')

merged.info()

已保存清洗数据: data_clean\merged_data.xlsx
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 684 entries, 0 to 683
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   city_code      684 non-null    object 
 1   city           684 non-null    object 
 2   year           684 non-null    int64  
 3   income         682 non-null    float64
 4   expend         682 non-null    float64
 5   gdp            682 non-null    float64
 6   deposit        679 non-null    float64
 7   re_invest      666 non-null    float64
 8   re_sales_area  665 non-null    float64
 9   re_avg_price   665 non-null    float64
dtypes: float64(7), int64(1), object(2)
memory usage: 53.6+ KB
